# Fine-tuning Laya for match_node

This notebook fine-tunes Laya on a small part of the train split of ZESHEL, step by step, and plots the curves of the training: the
loss in train and in val at every epoch. It uses the same code as `finetune.py` (the `Finetuner` and `curves.py`), so what it shows is
what a full run does. A full run takes hours: run it with the script (see the README) and plot its curves at the end of this notebook.

## The words of a training

- **Row / item / pair**: a row of the data is a pair (a mention and a known instance) with the expected answer (`same` is true or
  false). Laya turns every row into an item: the tokens of the state, the question and the target (the expected answer as
  probabilities of no and yes).
- **Epoch**: one pass over all the training items, in a new random order every time.
- **Micro-batch, gradient accumulation and effective batch**: the model reads `micro_batch` items at a time and adds up their
  gradients (the direction in which every weight should move to lower the loss) over `grad_accum` micro-batches before it moves the
  weights once (an **update**, or optimizer step). The effective batch is `micro_batch x grad_accum` (64): the items behind every
  update. A smaller micro-batch needs less memory for the same training.
- **Learning rate**: how far the weights move at every update. Laya lowers it along a cosine from its start to a minimum at the last
  update, so the last epochs move the weights less. That is why epoch 4 of a training of 6 epochs is not a training of 4 epochs.
- **Loss**: what the training lowers. Laya trains with `rlcd` (a term of reinforcement over noisy answers plus the cross-entropy),
  which can be negative, so it is not comparable with the loss of val. The curves use the **log-loss** (cross-entropy) in both:
  minus the logarithm of the probability the model gives the right answer. It is 0.69 for a model that always says 50%, and it
  punishes a confident wrong answer hard.
- **Train / val / test**: the model learns from train, we take decisions on val (the epochs, the variants, the threshold) and we
  look at test once at the end. The worlds of the val and test splits of ZESHEL are not in train: they measure whether the model
  **generalizes**.
- **Calibration slice and temperature**: Laya holds out 10% of the training items (400 at most) before training and, at the end,
  fits a temperature on them: a number that divides the logits so that a confidence of 0.8 is right 80% of the times.
- **Accuracy, mean confidence and ECE**: the share of right answers of the pairs, how sure the model says it is on average, and the
  Expected Calibration Error: how far its confidence is from its accuracy. A mean confidence above the accuracy is **overconfidence**.
- **Overfitting, underfitting and early stopping**: if the loss of train keeps going down while that of val stalls or goes up, the
  model memorizes train instead of learning what generalizes (overfitting); if both stay high, it has not learnt enough
  (underfitting). Early stopping keeps the epoch with the best val: `--keep-best` saves it.

## Setup

The notebook runs from the root of the repository, as the scripts do.

In [ ]:
import json, os, subprocess, sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.basename(os.getcwd()) == 'match_node' else os.getcwd()
os.chdir(ROOT)
if ROOT not in sys.path:
	sys.path.insert(0, ROOT)

import matplotlib.pyplot as plt

from training.match_node import curves, finetune

print(ROOT)

## A short training

The first `LIMIT` rows of the data, `EPOCHS` epochs and curve sets of `CURVE_MENTIONS` mentions. With these values it takes about
half an hour on Apple Silicon. `OUT` is the folder of the checkpoint.

In [ ]:
LIMIT		   = 2000
EPOCHS		   = 2
CURVE_MENTIONS = 100
DEVICE		   = 'auto'
OUT			   = os.path.join(finetune.OUT, 'notebook')

The configuration of the training: the defaults of Laya, with the epochs, a micro-batch of 4 (with 16 of them per update, the same
64 items) and gradient checkpointing (it computes the activations of the model again instead of keeping them: slower, but it needs
about 11 GB instead of 19 GB).

In [ ]:
from laya.train import TrainConfig

config = TrainConfig()
config.epochs				  = EPOCHS
config.micro_batch			  = 4
config.grad_accum			  = finetune.EFFECTIVE_BATCH//config.micro_batch
config.gradient_checkpointing = True

print('epochs %d, micro-batch %d x %d = %d items per update, encoder learning rate %g, loss %s'
	  % (config.epochs, config.micro_batch, config.grad_accum, config.micro_batch*config.grad_accum, config.encoder_lr, config.loss))

### 1. Load the base model

`convaiinnovations/laya`, the model of `MatchNodeLayaDecider`, on the device (`mps` on Apple Silicon).

In [ ]:
finetuner = finetune.Finetuner(finetune.BASE, config, DEVICE)

parameters = sum(p.numel() for p in finetuner.model.parameters())
print('%s on %s: %.0f million parameters, max_len %d' % (finetune.BASE, finetuner.device, parameters/1e6, finetuner.max_len))

### 2. Read the rows and turn them into items

A row is a pair: the state is what `MatchNodeLayaDecider` asks the model (the known instance and the mention, with its texts), and
`expected` is the answer.

In [ ]:
rows = finetune.read_rows(finetune.DATA, LIMIT)
finetuner.prepare(rows)

positives = sum(row['expected']['same'] for row in rows)
print('%d rows (%d positive), %d items: %d to train on and %d in the calibration slice'
	  % (len(rows), positives, len(finetuner.items), len(finetuner.train_index), len(finetuner.calib_items)))

print(json.dumps(rows[0]['state'], indent = 1)[:1500])

An item keeps the tokens of the state and the target: the expected answer as the probabilities of no and yes.

In [ ]:
item = finetuner.items[0]
print('%d tokens of state, target (no, yes) %s' % (len(item['state_ids']), item['target']))
print(finetuner.tok.decode(item['state_ids'][:80]))

### 3. The curve sets

The model is scored before the training and at the end of every epoch on whole mentions of train (that it trains on) and of val
(that it never sees).

In [ ]:
train_set = curves.create_train_set(finetuner, finetune.CURVE_TRAIN, CURVE_MENTIONS, finetune.CURVE_SEED)
val_set	  = curves.create_val_set(finetuner, finetune.CURVE_VAL, CURVE_MENTIONS, finetune.CURVE_SEED)
scorer	  = curves.EpochScorer(finetuner, [train_set, val_set], OUT)

for curve_set in scorer.curve_sets:
	print('%s: %d mentions, %d pairs' % (curve_set['name'], len(curve_set['mentions']), len(curve_set['items'])))

### 4. Train

It prints the `rlcd` loss of one micro-batch every 100 micro-batches (noisy: it is 4 items), and at the end of every epoch the
log-loss and the accuracy of both sets. With `best_folder`, the epoch with the lowest log-loss in val is saved in it.

In [ ]:
history = finetuner.train(scorer, os.path.join(OUT, 'best'))

### 5. Calibrate and save

The temperature is fitted on the calibration slice, and the checkpoint is saved in `OUT`, where `laya.load()` and
`MatchNodeLayaDecider` load it.

In [ ]:
calibrated = finetuner.calibrate()
finetuner.save(OUT, calibrated)

print('temperature', calibrated['fitted']['temperature'], '- saved in', OUT)

## The curves

`plot_curves` draws the history of one or more checkpoints: the log-loss and the accuracy (with the mean confidence, dashed) of train
and val at every epoch. Epoch 0 is the base model.

In [ ]:
def load_history(folder):
	with open(os.path.join(folder, 'history.json')) as f:
		return json.load(f)


def plot_metric(axis, history, metric, label, style = '-'):
	epochs = [entry['epoch'] for entry in history['epochs']]
	for name in history['sets']:
		values = [entry[name][metric] for entry in history['epochs']]
		axis.plot(epochs, values, style, marker = 'o', label = '%s %s%s' % (name, metric, label))


def plot_curves(*folders):
	figure, (loss_axis, accuracy_axis) = plt.subplots(1, 2, figsize = (12, 4))
	for folder in folders:
		history = load_history(folder)
		label	= '' if len(folders) == 1 else ' (%s)' % os.path.basename(os.path.normpath(folder))
		plot_metric(loss_axis, history, 'loss', label)
		plot_metric(accuracy_axis, history, 'accuracy', label)
		plot_metric(accuracy_axis, history, 'mean_confidence', label, style = '--')
		if history['best'] is not None:
			loss_axis.axvline(history['best']['epoch'], color = 'grey', linestyle = ':')

	loss_axis.set(title = 'log-loss (lower is better)', xlabel = 'epoch')
	accuracy_axis.set(title = 'accuracy and mean confidence (dashed)', xlabel = 'epoch')
	loss_axis.legend()
	accuracy_axis.legend()
	plt.show()


plot_curves(OUT)

The histogram of the scores of the pairs of val at an epoch, positives and negatives apart: a model that separates them well puts
the positives near 1 and the negatives near 0; a saturated one puts almost all of them at the extremes.

In [ ]:
def load_scores(folder, name, epoch):
	path = os.path.join(folder, 'curves', '%s__epoch_%d.jsonl' % (name, epoch))
	with open(path) as f:
		mentions = [json.loads(line) for line in f][1:]

	positives, negatives = [], []
	for mention in mentions:
		for candidate in mention['candidates']:
			if candidate['gold']:
				positives.append(candidate['score'])
			else:
				negatives.append(candidate['score'])

	return positives, negatives


def plot_scores(folder, epoch, name = 'val'):
	positives, negatives = load_scores(folder, name, epoch)
	plt.figure(figsize = (8, 3))
	plt.hist(negatives, bins = 50, range = (0, 1), alpha = 0.6, label = 'negative pairs (%d)' % len(negatives))
	plt.hist(positives, bins = 50, range = (0, 1), alpha = 0.6, label = 'positive pairs (%d)' % len(positives))
	plt.yscale('log')
	plt.title('scores of %s at epoch %d' % (name, epoch))
	plt.xlabel('probability of yes')
	plt.legend()
	plt.show()


plot_scores(OUT, 0)
plot_scores(OUT, EPOCHS)

## The decisions of every epoch

The scores of every epoch are saved as runs of the evaluation, so `evaluation/match_node/score.py` scores the decision of every
mention (linked, wrong link, missed) with any threshold:

In [ ]:
def score_epoch(folder, epoch, name = 'val', threshold = None):
	path	= os.path.join(folder, 'curves', '%s__epoch_%d.jsonl' % (name, epoch))
	command = [sys.executable, '-m', 'evaluation.match_node.score', path]
	if threshold is not None:
		command += ['--threshold', str(threshold)]

	output = subprocess.run(command, capture_output = True, text = True).stdout
	print('\n'.join(line for line in output.splitlines() if line.startswith(('Run', 'decision', 'best'))))


for epoch in range(EPOCHS + 1):
	score_epoch(OUT, epoch)

## The curves of a full run

A full run is trained with the script (it takes hours, so the Mac must not sleep):

```bash
caffeinate -i python -m training.match_node.finetune --epochs 6 --micro-batch 4 --checkpointing --keep-best
```

Set `RUN` to its folder to plot its curves; with several folders, `plot_curves` compares them.

In [ ]:
RUN = None		# e.g., os.path.join(finetune.OUT, 'zeshel_train__20261124-220000')

if RUN is not None:
	plot_curves(RUN)
	plot_scores(RUN, load_history(RUN)['epochs'][-1]['epoch'])